# PathSnap: train models from candidate CSV data

This notebook trains **new models in this Colab runtime** from the supplied CSV tables. The CSVs were prepared once from the original 80 training and 10 student-test trips using ordered route alignment and the project's candidate feature code. Only training trips are used to fit models; student-test labels are used for the final comparison.

Upload `PathSnap_Prepared_CSV.zip` to Colab Files and run all cells. The final outputs and saved `joblib` files are evidence of a fresh run. The CSVs skip expensive graph feature preparation. They do **not** speed up full Viterbi decoder tuning, and this short notebook reports pointwise candidate selection rather than route metrics. Use `PathSnap_Colab.ipynb` for the complete decoder experiment.

In [ ]:
%pip -q install lightgbm==4.7.0
import hashlib, json, time
from pathlib import Path
from zipfile import ZipFile
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from sklearn.ensemble import HistGradientBoostingClassifier

data_zip = Path('/content/PathSnap_Prepared_CSV.zip')
if not data_zip.is_file():
    raise FileNotFoundError('Upload PathSnap_Prepared_CSV.zip to Colab Files, then rerun.')
with ZipFile(data_zip) as archive:
    archive.extractall('/content')
data_dir = Path('/content/PathSnap_CSV_Data')
manifest = json.loads((data_dir / 'MANIFEST.json').read_text())
for split in ('train', 'student_test'):
    for suffix, key in [('candidates', 'sha256'), ('observations', 'observation_sha256')]:
        path = data_dir / f'{split}_{suffix}.csv.gz'
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        assert actual == manifest['partitions'][split][key], path
print('CSV checksums passed')
display(pd.DataFrame(manifest['partitions']).T[['trips', 'observations', 'candidates', 'positive_rows']])

In [ ]:
features = manifest['columns'][5:]
types = {'trip_id': str, 'observation_index': 'int32', 'arc_id': 'int32',
         'positive_label': 'int8', 'alignment_distance_m': 'float32'}
types.update({f: 'float32' for f in features})
train = pd.read_csv(data_dir / 'train_candidates.csv.gz', dtype=types)
test = pd.read_csv(data_dir / 'student_test_candidates.csv.gz', dtype=types)
test_obs = pd.read_csv(data_dir / 'student_test_observations.csv.gz', dtype={
    'trip_id': str, 'observation_index': 'int32', 'reference_arc_id': 'int32'})
assert train.trip_id.nunique() == 80 and test.trip_id.nunique() == 10
assert len(test_obs) == 19160
assert not set(train.trip_id) & set(test.trip_id)
print('Loaded full training candidate rows:', len(train))
print('Loaded held-out test candidate rows:', len(test))
print('Feature columns:', len(features))
display(train[['trip_id', 'observation_index', 'arc_id', 'positive_label', *features[:4]]].head())

In [ ]:
# Match the reviewed protocol: fit on trusted observations with a correct candidate.
key = ['trip_id', 'observation_index']
has_positive = train.groupby(key, sort=False).positive_label.transform('max').eq(1)
fit_rows = train.loc[(train.alignment_distance_m <= 75) & has_positive].copy()
groups = fit_rows.groupby(key, sort=False).size().to_numpy()
assert groups.sum() == len(fit_rows)
assert fit_rows.groupby(key, sort=False).positive_label.max().eq(1).all()
X_train = fit_rows[features].to_numpy(dtype=np.float32, copy=False)
y_train = fit_rows.positive_label.to_numpy(dtype=np.int8)
X_test = test[features].to_numpy(dtype=np.float32, copy=False)
print('Fit observations:', len(groups), 'Fit candidate rows:', len(fit_rows))
print('Held-out test observations:', len(test_obs))

In [ ]:
# Three fresh fits; tuned settings came from training-trip optimization in the reviewed project.
common = dict(objective='lambdarank', max_bin=63, n_jobs=4, verbosity=-1, random_state=42)
configs = {
    'LightGBM untuned': dict(n_estimators=90, num_leaves=31, max_depth=-1,
                              learning_rate=.08, min_child_samples=40, reg_lambda=0.),
    'LightGBM tuned': dict(n_estimators=110, num_leaves=20, max_depth=4,
                            learning_rate=.13687258703610514, min_child_samples=117,
                            reg_lambda=2.2225433823808736),
}
models = {}
scores = {}
for name, config in configs.items():
    started = time.time()
    model = lgb.LGBMRanker(**(common | config))
    model.fit(X_train, y_train, group=groups)
    models[name] = model
    scores[name] = model.predict(X_test)
    print(f'{name} trained on {len(fit_rows):,} rows in {time.time()-started:.1f}s')

started = time.time()
hgb = HistGradientBoostingClassifier(max_iter=48, max_bins=63,
    l2_regularization=.3, min_samples_leaf=50, max_depth=7,
    early_stopping=False, random_state=42)
hgb.fit(X_train, y_train)
models['HistGradientBoosting'] = hgb
scores['HistGradientBoosting'] = hgb.predict_proba(X_test)[:, 1]
print(f'HistGradientBoosting trained on {len(fit_rows):,} rows in {time.time()-started:.1f}s')

In [ ]:
distance = test.distance_m.to_numpy()
heading = test.heading_cos.to_numpy()
movement = test.speed_mps.to_numpy() * test.elapsed_s.to_numpy()
scores['Nearest road'] = -distance / 12.
scores['Distance + heading'] = -(distance + np.where(movement >= 4., 8. * (1. - heading), 0.)) / 12.

def evaluate(name, values):
    frame = test[['trip_id', 'observation_index', 'arc_id', 'positive_label']].copy()
    frame['score'] = values
    best = frame.loc[frame.groupby(key, sort=False).score.idxmax()]
    # Full denominator includes observations with zero candidates.
    matched = test_obs[key].merge(best[key + ['positive_label']], on=key, how='left')
    has_reference_candidate = test.groupby(key, sort=False).positive_label.max().reset_index()
    coverage = test_obs[key].merge(has_reference_candidate, on=key, how='left')
    return {'method': name, 'test_observations': len(matched),
            'candidate_misses': int((coverage.positive_label.fillna(0) == 0).sum()),
            'correct': int(matched.positive_label.fillna(0).sum()),
            'point_accuracy': float(matched.positive_label.fillna(0).mean())}

comparison = pd.DataFrame([evaluate(name, values) for name, values in scores.items()])
comparison = comparison.sort_values('point_accuracy', ascending=False)
display(comparison.style.format({'point_accuracy': '{:.2%}'}))
ax = comparison.sort_values('point_accuracy').plot.barh(x='method', y='point_accuracy',
    legend=False, color='#2457d6', figsize=(9, 4.5))
ax.set(title='Fresh Colab training: held-out student-test point accuracy', xlabel='Accuracy', ylabel='', xlim=(0, 1))
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(axis='x', alpha=.2)
plt.tight_layout()
plt.show()

In [ ]:
out = Path('/content/PathSnap_CSV_Training_Output')
out.mkdir(exist_ok=True)
comparison.to_csv(out / 'fresh_model_comparison.csv', index=False)
for name, model in models.items():
    path = out / (name.lower().replace(' ', '_') + '.joblib')
    joblib.dump(model, path, compress=3)
    print('Saved', path.name, path.stat().st_size, 'bytes')
print('Results CSV:', out / 'fresh_model_comparison.csv')
print('This notebook fitted models in this runtime; decoder metrics require the full project notebook.')